# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samhere3116-maker/ML-Pipeline/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1:
Hypothesis: pages not updated in 180+ days decline more than recently updated ones.

In [9]:
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
QUERY90 = f"read_parquet('{REL}/fact_content_query_90d.parquet')"
MONTH_START, MONTH_END = '2026-03-01', '2026-03-31'

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
staleness = con.sql(f"""
    WITH windowed AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date > DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_last30,
            SUM(CASE WHEN report_date <= DATE '{MONTH_END}' - INTERVAL 30 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev30
        FROM {FACT}
        WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50
    ),
    with_age AS (
        SELECT w.*, DATE_DIFF('day', d.content_updated_date, DATE '{MONTH_END}') AS days_since_update
        FROM windowed w
        JOIN {DIM_CONTENT} d USING (content_hash_id)
        WHERE d.content_updated_date IS NOT NULL
    ),
    labeled AS (
        SELECT *,
            CASE WHEN imp_last30 < 0.8 * imp_prev30 THEN 1 ELSE 0 END AS is_declining,
            CASE WHEN days_since_update >= 180 THEN 'stale' ELSE 'fresh' END AS staleness_bucket
        FROM with_age
    )
    SELECT staleness_bucket, COUNT(*) AS n, AVG(is_declining) AS decline_rate
    FROM labeled
    GROUP BY staleness_bucket
""").df()
staleness

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,staleness_bucket,n,decline_rate
0,fresh,29922,0.000535
1,stale,3,0.000000


Signal 2 — CTR vs position:
Hypothesis: even at similar ranking positions, CTR varies a lot — meaning "well-ranked but under-clicked" pages genuinely exist.

In [11]:
ctr_position = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks, SUM(gsc_impressions) AS impressions,
            AVG(gsc_avg_position) AS avg_position
        FROM {FACT}
        WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING impressions >= 50
    ),
    tiered AS (
        SELECT *, clicks * 1.0 / impressions AS ctr,
            CASE WHEN avg_position <= 10 THEN 'top10' WHEN avg_position <= 20 THEN '11-20' ELSE '20+' END AS position_tier
        FROM agg
    )
    SELECT position_tier, COUNT(*) AS n, AVG(ctr) AS avg_ctr, MIN(ctr) AS min_ctr, MAX(ctr) AS max_ctr
    FROM tiered
    GROUP BY position_tier
    ORDER BY position_tier
""").df()
ctr_position

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,position_tier,n,avg_ctr,min_ctr,max_ctr
0,11-20,24294,0.002390,0.0,0.111111
1,20+,29915,0.001277,0.0,0.155844
2,top10,61905,0.003354,0.0,0.162500


Signal 1 (staleness, flag-linked): FALSE. Only 3 of 29,925 pages qualify as "stale" (180+ days since update), too few to test, and directionally the opposite of expected. This suggests content_updated_date doesn't reflect real editorial activity in this data, so a staleness-based rule isn't usable here.
Signal 2 (CTR vs position): CONFIRMED. Average CTR drops from 0.34% (top 10) to 0.13% (20+), as expected. But CTR varies widely within each tier (0% to 11-16%), meaning position alone doesn't explain CTR — some well-ranked pages are genuinely under-clicked. That gap is what my rule targets.

My rule: flag a page if it's ranked in the top 20 on average AND its CTR sits meaningfully below its tier's average CTR, with enough impressions that the gap isn't noise. Reason code: low_ctr_visible_page. Action: review_title_and_meta.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
scored = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks, SUM(gsc_impressions) AS impressions,
            AVG(gsc_avg_position) AS avg_position
        FROM {FACT}
        WHERE report_date BETWEEN '{MONTH_START}' AND '{MONTH_END}'
        GROUP BY 1, 2
        HAVING impressions >= 50 AND avg_position <= 20
    ),
    tiered AS (
        SELECT *, clicks * 1.0 / impressions AS ctr,
            CASE WHEN avg_position <= 10 THEN 'top10' ELSE '11-20' END AS position_tier
        FROM agg
    ),
    tier_avgs AS (
        SELECT position_tier, AVG(ctr) AS tier_avg_ctr FROM tiered GROUP BY position_tier
    )
    SELECT t.client_hash_id, t.content_hash_id, t.impressions, t.avg_position, t.ctr,
           t.position_tier, ta.tier_avg_ctr,
           (ta.tier_avg_ctr - t.ctr) * t.impressions AS baseline_action_score,
           'low_ctr_visible_page' AS reason_code,
           'review_title_and_meta' AS action
    FROM tiered t
    JOIN tier_avgs ta USING (position_tier)
    WHERE t.ctr < ta.tier_avg_ctr
    ORDER BY baseline_action_score DESC
""").df()

print(f"{len(scored):,} pages flagged")
import os
os.makedirs("work/outputs", exist_ok=True)
scored.to_csv("work/outputs/baseline_action_score.csv", index=False)
scored.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

58,101 pages flagged


,client_hash_id,content_hash_id,impressions,avg_position,ctr,position_tier,tier_avg_ctr,baseline_action_score,reason_code,action
0,client_23a62021009f63c4,content_44f34c0a90047651,212404.0,7.346909,0.000113,top10,0.003354,688.408864,low_ctr_visible_page,review_title_and_meta
1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,4.545582,0.000007,top10,0.003354,451.740052,low_ctr_visible_page,review_title_and_meta
2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,3.219473,0.000301,top10,0.003354,436.689663,low_ctr_visible_page,review_title_and_meta
3,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075.0,9.385150,0.000008,top10,0.003354,415.150966,low_ctr_visible_page,review_title_and_meta
4,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,2.563756,0.001420,top10,0.003354,393.534540,low_ctr_visible_page,review_title_and_meta
5,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,5.789019,0.000626,top10,0.003354,361.720572,low_ctr_visible_page,review_title_and_meta
6,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,9.536301,0.000139,top10,0.003354,345.839698,low_ctr_visible_page,review_title_and_meta
7,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,3.361195,0.001534,top10,0.003354,310.894734,low_ctr_visible_page,review_title_and_meta
8,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,7.786219,0.000045,top10,0.003354,295.621987,low_ctr_visible_page,review_title_and_meta
9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,4.450106,0.001858,top10,0.003354,290.811648,low_ctr_visible_page,review_title_and_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

content_44f34c0a90047651 (client_23a62021009f63c4): Action: review_title_and_meta. Ranks 7.3 on average with 212,404 impressions but only 0.011% CTR against a 0.33% tier average — huge visibility, almost no clicks. Would be wrong if the title is already accurate and the real issue is search intent mismatch rather than the title itself.

content_8e1334d6356668e3 (client_73cda7b4e4f265ea): Action: review_title_and_meta. 134,984 impressions at position 4.5, CTR of just 0.0007%, essentially zero. Would be wrong if a rich result or featured snippet from a competitor is absorbing the clicks regardless of title quality.

content_34a70fea29d15f24 (client_62f4a7e64f5e0096): Action: review_title_and_meta. Strong position (3.2) with 143,019 impressions but CTR of 0.03%. Would be wrong if this page recently changed URLs or titles and Google hasn't fully re-indexed the update yet.

content_fec55986a1868d62 (client_73cda7b4e4f265ea): Action: review_title_and_meta. Position 9.4, 124,075 impressions, CTR 0.0008%. Would be wrong if the query itself is informational/navigational and users don't click through regardless of title (e.g. a "what is X" query answered directly in the snippet).

content_8d7d99f109e19aa2 (client_e547b89c05043229): Action: review_title_and_meta. Very strong position (2.6) with 203,497 impressions but CTR only 0.14%. Would be wrong if this is a seasonal page and the dip is temporary rather than a real title problem.

content_7c6373141eae744a (client_62f4a7e64f5e0096): Action: review_title_and_meta. Position 5.8, 132,593 impressions, CTR 0.06%. Would be wrong if duplicate/similar content on the same site is cannibalizing clicks that should go to this page.

content_f6116743b00afc2d (client_62f4a7e64f5e0096): Action: review_title_and_meta. Position 9.5, 107,584 impressions, CTR 0.014%. Would be wrong if the meta description (not the title) is the actual problem, which this rule can't distinguish.

content_acbcc847f8996314 (client_62f4a7e64f5e0096): Action: review_title_and_meta. Position 3.4, 170,808 impressions, CTR 0.15%. Would be wrong if this page already ranks for a branded query where users recognize the brand and click a different, more familiar result instead.

content_cd3d932d4e1c8db0 (client_9958f0a7ae1df715): Action: review_title_and_meta. Position 7.8, 89,332 impressions, CTR 0.005%. Would be wrong if a broken link or redirect issue is suppressing clicks rather than a weak title.

content_b99ea6861864dea5 (client_62f4a7e64f5e0096): Action: review_title_and_meta. Position 4.5, 194,337 impressions, CTR 0.19%. Would be wrong if this page is genuinely fine and just gets a lot of "zero-click" impressions from a broad, ambiguous keyword.

content_f43118e089ecc69a (client_73cda7b4e4f265ea): Action: review_title_and_meta. Position 5.0, 139,417 impressions, CTR 0.14%. Would be wrong if it's a PDF or non-HTML result type that displays differently in search than a normal page.

content_046fc480045b88f5 (client_a80fca3f171ed1de): Action: review_title_and_meta. Position 7.3, 83,788 impressions, CTR 0.009%. Would be wrong if the impressions are inflated by an unrelated keyword the page barely relates to.

content_9540d884af3e41fd (client_a80fca3f171ed1de): Action: review_title_and_meta. Position 7.8, 82,376 impressions, CTR 0.016%. Would be wrong if it's a very recent page still stabilizing in rankings, where CTR hasn't settled yet.

content_425715547c6a3ea8 (client_73cda7b4e4f265ea): Action: review_title_and_meta. Position 6.4, 71,513 impressions, CTR 0.006%. Would be wrong if the title/meta were already updated recently and this data predates that change.

content_306bc78dff1eb683 (client_e547b89c05043229): Action: review_title_and_meta. Very strong position (1.5) but CTR only 0.043% against 80,821 impressions — unusual for a #1-2 spot. Would be wrong if a "People also ask" or knowledge panel is absorbing clicks ahead of this result.

content_e578ac84778da489 (client_73cda7b4e4f265ea): Action: review_title_and_meta. Position 4.1, 117,764 impressions, CTR 0.14%. Would be wrong if this page targets a highly competitive query where even top results naturally get lower CTR due to many similar options.

content_36fc1ee501ec072d (client_62f4a7e64f5e0096): Action: review_title_and_meta. Position 6.5, 73,135 impressions, CTR 0.022%. Would be wrong if this page's traffic is mostly from a country/language segment where display and click behavior differ from the norm.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The clearest weak spot in this queue is client concentration: client_62f4a7e64f5e0096 and client_73cda7b4e4f265ea each appear 5 times in the top 17, while most other clients appear once or not at all. Because my score multiplies the CTR gap by raw impression volume, clients with naturally higher search volume dominate the ranking regardless of how severe their CTR problem actually is relative to their own baseline. A fairer version would normalize the gap within each client, not just within each position tier, so smaller clients with equally real problems aren't crowded out. On leakage: this score uses only gsc_clicks, gsc_impressions, and gsc_avg_position from within the March window itself — no imp_last30, no is_declining label, and no FlyRank product flags were used as inputs.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.